# Détecteur de discours haineux — Colab

Prototype pédagogique avec données fictives. L'objectif est de classer des messages et de recommander une révision humaine quand la confiance est faible.


In [1]:
!pip install -q pandas scikit-learn matplotlib joblib


In [5]:
import pandas as pd
from io import StringIO

csv_data = '''message,label
"Merci pour ton aide, ton explication est très claire.",acceptable
"Je ne suis pas d'accord avec toi, mais je respecte ton point de vue.",acceptable
"Peux-tu partager la source de cette information ?",acceptable
"Cette proposition mérite une discussion plus calme.",acceptable
"Bravo à toute l'équipe pour ce travail.",acceptable
"Ton idée est mauvaise et tu refuses d'écouter.",insulte
"Tu racontes vraiment n'importe quoi.",insulte
"Arrête de faire ton intéressant, c'est pathétique.",insulte
"Tu es incapable de comprendre le sujet.",insulte
"On va te suivre partout sur ce forum pour te faire honte.",harcelement
"Chaque fois que tu publieras, on viendra t'humilier.",harcelement
"Tout le monde devrait t'ignorer et te ridiculiser en boucle.",harcelement
"Les personnes de ce groupe ne devraient pas avoir leur place ici.",haine
"Ce groupe est inférieur et devrait être exclu de nos espaces.",haine
"On devrait refuser les services aux gens de cette origine.",haine
"Les personnes comme elles sont toutes dangereuses.",haine
"Je vais venir te faire du mal si tu continues.",menace
"Tu vas regretter physiquement ce que tu viens de dire.",menace
"Je sais où te trouver et je vais m'en prendre à toi.",menace
"Continue et je vais te casser la figure.",menace
'''
df = pd.read_csv(StringIO(csv_data))
df.head()

,message,label
0,"Merci pour ton aide, ton explication est très ...",acceptable
1,"Je ne suis pas d'accord avec toi, mais je resp...",acceptable
2,Peux-tu partager la source de cette information ?,acceptable
3,Cette proposition mérite une discussion plus c...,acceptable
4,Bravo à toute l'équipe pour ce travail.,acceptable


In [6]:
import re, unicodedata
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

def normalize_text(text):
    text = str(text).lower().strip()
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'https?://\S+', ' lien ', text)
    text = re.sub(r'@\w+', ' mention ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

# Drop rows where 'label' is NaN before splitting the data
df_cleaned = df.dropna(subset=['label']).copy()
X_train, X_test, y_train, y_test = train_test_split(df_cleaned['message'], df_cleaned['label'], test_size=0.25, random_state=42, stratify=df_cleaned['label'])
model = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=normalize_text, ngram_range=(1,2), max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred, zero_division=0))

              precision    recall  f1-score   support

  acceptable       0.50      1.00      0.67         1
       haine       1.00      1.00      1.00         1
 harcelement       0.00      0.00      0.00         1
     insulte       0.00      0.00      0.00         1
      menace       1.00      1.00      1.00         1

    accuracy                           0.60         5
   macro avg       0.50      0.60      0.53         5
weighted avg       0.50      0.60      0.53         5



In [7]:
def predict_message(message, threshold=0.62):
    probs = model.predict_proba([message])[0]
    classes = list(model.classes_)
    best = int(np.argmax(probs))
    label = classes[best]
    confidence = float(probs[best])
    return {
        'message': message,
        'label': label,
        'confidence': confidence,
        'revision_humaine_recommandee': confidence < threshold,
        'probabilities': dict(zip(classes, map(float, probs)))
    }

predict_message('Je ne suis pas d’accord, mais discutons calmement.')


{'message': 'Je ne suis pas d’accord, mais discutons calmement.',
 'label': 'acceptable',
 'confidence': 0.2811731128394332,
 'revision_humaine_recommandee': True,
 'probabilities': {'acceptable': 0.2811731128394332,
  'haine': 0.19106599707441055,
  'harcelement': 0.16157263022334206,
  'insulte': 0.17931810328972328,
  'menace': 0.1868701565730909}}

In [ ]:
message = input('Écris un message à analyser : ')
predict_message(message)


## Note éthique
Ce prototype peut se tromper. Il faut éviter toute sanction automatique. Les cas incertains doivent être relus par une personne, avec possibilité de contestation.
